# Subset Road TTS Robustness: 2 Iterations vs 5 Iterations

This notebook reproduces the TTS robustness plots for the road subset comparison.
It uses the common cases between the saved 2-iteration run and the current 5-iteration run:

- Developments: 28, 109, 254, 267
- Scenarios: scenario_19, scenario_26, scenario_44, scenario_64, scenario_78

The central plot mirrors the original `mean TTS with standard deviation across scenarios` plot, but compares `2iter` and `5iter` instead of Road/Rail modes.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

OUTPUT_DIR = Path("/cluster/home/lkuehner/MSc_Thesis/infraScan/infraScanIntegrated/outputs/compare_2iter_vs_5iter_current")
TTS_COMPONENTS = OUTPUT_DIR / "tts_components_common_cases.csv"
WEIGHTED_TT = OUTPUT_DIR / "weighted_tt_common_cases.csv"
SUMMARY_BY_DEV = OUTPUT_DIR / "weighted_tt_summary_by_dev.csv"

PLOT_DIR = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", TTS_COMPONENTS)
print("Plot output:", PLOT_DIR)


Input: /cluster/home/lkuehner/MSc_Thesis/infraScan/infraScanIntegrated/outputs/compare_2iter_vs_5iter_current/tts_components_common_cases.csv
Plot output: /cluster/home/lkuehner/MSc_Thesis/infraScan/infraScanIntegrated/outputs/compare_2iter_vs_5iter_current/plots


In [ ]:
tts_components = pd.read_csv(TTS_COMPONENTS)
weighted_tt = pd.read_csv(WEIGHTED_TT)
summary_by_dev = pd.read_csv(SUMMARY_BY_DEV)

for df in [tts_components, weighted_tt, summary_by_dev]:
    df.columns = df.columns.str.strip()

display(tts_components.head())
display(summary_by_dev)


In [ ]:
def build_subset_tts_minutes_frame(tts_components: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for iteration in ["2iter", "5iter"]:
        suffix = f"_{iteration}"
        tmp = tts_components[[
            "development",
            "scenario",
            f"tt_savings_peak{suffix}",
            f"network_savings{suffix}",
            f"origin_access_savings{suffix}",
            f"destination_access_savings{suffix}",
            f"monetized_savings{suffix}",
        ]].copy()
        tmp["iteration"] = iteration
        tmp["mode"] = "Road"
        tmp = tmp.rename(columns={
            f"tt_savings_peak{suffix}": "tts_minutes",
            f"network_savings{suffix}": "network_minutes",
            f"origin_access_savings{suffix}": "origin_access_minutes",
            f"destination_access_savings{suffix}": "destination_access_minutes",
            f"monetized_savings{suffix}": "tts_chf",
        })
        frames.append(tmp)
    return pd.concat(frames, ignore_index=True)


def build_tts_robustness(tts_minutes_frame: pd.DataFrame) -> pd.DataFrame:
    metrics = (
        tts_minutes_frame.groupby(["iteration", "mode", "development"], as_index=False)
        .agg(
            scenario_count=("scenario", "nunique"),
            median_tts_minutes=("tts_minutes", "median"),
            mean_tts_minutes=("tts_minutes", "mean"),
            std_tts_minutes=("tts_minutes", "std"),
            min_tts_minutes=("tts_minutes", "min"),
            max_tts_minutes=("tts_minutes", "max"),
            q1_tts_minutes=("tts_minutes", lambda values: values.quantile(0.25)),
            q3_tts_minutes=("tts_minutes", lambda values: values.quantile(0.75)),
        )
    )
    metrics["iqr_tts_minutes"] = metrics["q3_tts_minutes"] - metrics["q1_tts_minutes"]
    metrics["abs_median_tts_minutes"] = metrics["median_tts_minutes"].abs()
    metrics["relative_iqr_tts"] = metrics["iqr_tts_minutes"] / metrics["abs_median_tts_minutes"].replace(0, np.nan)
    metrics["relative_std_tts"] = metrics["std_tts_minutes"] / metrics["abs_median_tts_minutes"].replace(0, np.nan)
    return metrics.sort_values(["iteration", "abs_median_tts_minutes"], ascending=[True, False])


tts_minutes_frame = build_subset_tts_minutes_frame(tts_components)
tts_robustness = build_tts_robustness(tts_minutes_frame)

display(tts_minutes_frame.head())
display(tts_robustness)


In [ ]:
# Main plot: same idea as the copied plot, but 2iter vs 5iter for the Road subset.
tts_robustness_plot = tts_robustness.copy()
tts_robustness_plot["mean_tts_hours"] = tts_robustness_plot["mean_tts_minutes"] / 60.0
tts_robustness_plot["std_tts_hours"] = tts_robustness_plot["std_tts_minutes"] / 60.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
for ax, iteration in zip(axes, ["2iter", "5iter"]):
    subset = tts_robustness_plot[tts_robustness_plot["iteration"] == iteration].copy()
    subset = subset.sort_values("mean_tts_hours", ascending=False).reset_index(drop=True)
    y_pos = np.arange(len(subset))
    lower = subset["mean_tts_hours"] - subset["std_tts_hours"].fillna(0)
    upper = subset["mean_tts_hours"] + subset["std_tts_hours"].fillna(0)

    ax.hlines(y=y_pos, xmin=lower, xmax=upper, color="#9ecae1", linewidth=3.0, alpha=0.9)
    ax.scatter(subset["mean_tts_hours"], y_pos, color="#08519c", s=46, zorder=3)
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(subset["development"].astype(str), fontsize=10)
    ax.invert_yaxis()
    ax.set_title(f"Road subset {iteration}: mean TTS with std across scenarios")
    ax.set_xlabel("Travel time savings [hours]")
    ax.set_ylabel("Development ID")
    ax.grid(False)
    ax.grid(True, axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(PLOT_DIR / "subset_road_mean_tts_with_std_2iter_vs_5iter.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Median-vs-std diagnostic, useful for seeing whether 5iter reduces scenario sensitivity.
plot_df = tts_robustness.copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, iteration in zip(axes, ["2iter", "5iter"]):
    subset = plot_df[plot_df["iteration"] == iteration].copy()
    sns.scatterplot(
        data=subset,
        x="median_tts_minutes",
        y="std_tts_minutes",
        size="abs_median_tts_minutes",
        hue="abs_median_tts_minutes",
        palette="viridis",
        sizes=(80, 450),
        alpha=0.85,
        ax=ax,
        legend=False,
    )
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
    ax.set_title(f"Road subset {iteration}: median TTS vs standard deviation")
    ax.set_xlabel("Median TTS [minutes]")
    ax.set_ylabel("Standard deviation of TTS [minutes]")
    ax.grid(False)
    for _, row in subset.iterrows():
        ax.annotate(str(row["development"]), (row["median_tts_minutes"], row["std_tts_minutes"]), xytext=(5, 5), textcoords="offset points", fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "subset_road_median_tts_vs_std_2iter_vs_5iter.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Boxplot of TTS across the five common scenarios for each development and iteration.
box_df = tts_minutes_frame.copy()
box_df["development"] = box_df["development"].astype(str)
order = (
    tts_robustness[tts_robustness["iteration"] == "5iter"]
    .sort_values("median_tts_minutes", ascending=False)["development"]
    .astype(str)
    .tolist()
)
box_df["development"] = pd.Categorical(box_df["development"], categories=order, ordered=True)

plt.figure(figsize=(10, 5))
sns.boxplot(data=box_df, x="development", y="tts_minutes", hue="iteration", palette=["#fdae6b", "#9ecae1"])
plt.axhline(0, color="black", linewidth=1)
plt.title("Road subset: TTS distribution across common scenarios")
plt.xlabel("Development")
plt.ylabel("TTS [minutes]")
plt.tight_layout()
plt.savefig(PLOT_DIR / "subset_road_tts_boxplot_2iter_vs_5iter.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Optional: component means show that the iteration effect comes from network savings, not access/egress.
component_means = (
    tts_minutes_frame.groupby(["iteration", "development"], as_index=False)
    .agg(
        mean_total_tts=("tts_minutes", "mean"),
        mean_network=("network_minutes", "mean"),
        mean_origin_access=("origin_access_minutes", "mean"),
        mean_destination_access=("destination_access_minutes", "mean"),
    )
)
component_long = component_means.melt(
    id_vars=["iteration", "development"],
    value_vars=["mean_network", "mean_origin_access", "mean_destination_access"],
    var_name="component",
    value_name="mean_minutes",
)

plt.figure(figsize=(12, 5))
sns.barplot(data=component_long, x="development", y="mean_minutes", hue="component")
plt.axhline(0, color="black", linewidth=1)
plt.title("Road subset: mean TTS components across common scenarios")
plt.xlabel("Development")
plt.ylabel("Mean component [minutes]")
plt.tight_layout()
plt.savefig(PLOT_DIR / "subset_road_tts_component_means.png", dpi=200, bbox_inches="tight")
plt.show()

display(component_means)
